In [1]:
import torch
print(torch.cuda.is_available())

True


In [5]:
# torch.nn은 신경망 구성에 필요한 모든 요소 제공
# PyTorch의 모든 모듈을 nn.Module의 subclass

import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [6]:
# 학습을 위한 장치 얻기

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [99]:
# 클래스 정의하기

# NN 모델을 nn.Moudule의 하위클래스로 정의하고
# __init__에서 신경만 계층들을 초기화
# nn.Module을 상속받은 모든 클래스는 forward 메소드에 입력 데이터에 대한 연산들을 구현

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
        
# 정의한 모델 객체 선언하면 알아서 __init__ 적용
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [109]:
# 모델을 사용하기 위해 입력 데이터를 전달
# 이 때 일부 백그라운드 연산들과 함께 모델의 forward를 실행
# model.forward()를 직접 호출하면 안됨

X = torch.rand(1, 28, 28, device = device)
logits = model(X)
print(logits)
# 반환값에는 2차원 텐서 (dim = 0은 행, dim = 1은 열로 생각)
pred_probab = nn.Softmax(dim=1)(logits)
# dim=1로 softmax를 해야 결과값에 대해 훑을 수 있음
print(pred_probab)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

tensor([[ 0.0158,  0.0493, -0.0527, -0.0261,  0.1197, -0.0469, -0.0371, -0.0077,
          0.0175, -0.0804]], device='cuda:0', grad_fn=<AddmmBackward0>)
tensor([[0.1019, 0.1054, 0.0952, 0.0977, 0.1131, 0.0957, 0.0967, 0.0996, 0.1021,
         0.0926]], device='cuda:0', grad_fn=<SoftmaxBackward0>)
Predicted class: tensor([4], device='cuda:0')


In [110]:
# 모델 계층(Layer)

# FashionMNIST 28x28 크기 이미지 3개짜리 minibatch 가정

input_image = torch.rand(3,28,28)
print(input_image.size())


torch.Size([3, 28, 28])


In [111]:
# nn.Flatten

# 이 때 dim=0의 minibatch dimmension은 유지됨

flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [112]:
# nn.Linear

layer1 = nn.Linear(in_features = 28*28, out_features = 20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [113]:
# nn.ReLU

print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.0476, -0.0307,  0.2130, -0.1886, -0.4795, -0.2129, -0.1809,  0.3600,
         -0.2773,  0.3390, -0.0764, -0.4156, -0.4057,  0.1460, -0.0981, -0.8081,
          0.3466, -0.1724, -0.1092,  0.3370],
        [-0.1520,  0.0785,  0.5335, -0.2332, -0.1933,  0.1390,  0.0313,  0.2509,
          0.1377,  0.5013, -0.4447, -0.3973, -0.4119,  0.0888, -0.1403, -0.3714,
          0.4273,  0.1014, -0.0994,  0.9178],
        [-0.2138,  0.3519,  0.3011, -0.4289,  0.0213, -0.4404, -0.0798,  0.3409,
         -0.0368,  0.6835, -0.3757, -0.1019,  0.1215, -0.0837, -0.4317, -0.4266,
          0.4586,  0.0709, -0.2474,  0.5087]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0476, 0.0000, 0.2130, 0.0000, 0.0000, 0.0000, 0.0000, 0.3600, 0.0000,
         0.3390, 0.0000, 0.0000, 0.0000, 0.1460, 0.0000, 0.0000, 0.3466, 0.0000,
         0.0000, 0.3370],
        [0.0000, 0.0785, 0.5335, 0.0000, 0.0000, 0.1390, 0.0313, 0.2509, 0.1377,
         0.5013, 0.0000, 0.0000, 0.0000, 0.0888, 0.00

In [117]:
# nn.Sequential

# 순서를 갖는 모듈의 컨테이너
# 데이터는 정의된 것과 같은 순서로 모든 모듈들을 통해 전달

seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20,10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)
print(logits)

tensor([[-0.0109, -0.0745,  0.0743, -0.3453, -0.1422, -0.0642, -0.1996,  0.0615,
          0.2515, -0.0293],
        [ 0.0205, -0.1577,  0.0523, -0.3041, -0.0618, -0.1680, -0.0804,  0.0975,
          0.1729, -0.0477],
        [-0.0447, -0.1504, -0.0293, -0.2966, -0.0715, -0.1203, -0.1160,  0.1179,
          0.2451,  0.0938]], grad_fn=<AddmmBackward0>)


In [119]:
# nn.Softmax

# [0,1] 범위로 비례하여 scale, dim 매개변수는 값의 합이 1이 되는 차원을 나타냄

softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)
print(pred_probab)

tensor([[0.1025, 0.0962, 0.1116, 0.0734, 0.0899, 0.0972, 0.0849, 0.1102, 0.1333,
         0.1007],
        [0.1061, 0.0888, 0.1095, 0.0767, 0.0977, 0.0879, 0.0959, 0.1146, 0.1236,
         0.0991],
        [0.0982, 0.0883, 0.0997, 0.0763, 0.0956, 0.0910, 0.0914, 0.1155, 0.1312,
         0.1128]], grad_fn=<SoftmaxBackward0>)


In [120]:
# 모델 매개변수

# 신경망 내부의 많은 계층들은 parameterize됨
# parameters() 및 named_parameters() 메소드로 모든 매개변수에 접근 가능

print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name}|Size: {param.size()}|Values: {param[:2]}\n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight|Size: torch.Size([512, 784])|Values: tensor([[ 0.0350, -0.0321, -0.0037,  ..., -0.0037, -0.0334,  0.0338],
        [-0.0262,  0.0129, -0.0323,  ..., -0.0342, -0.0087, -0.0010]],
       device='cuda:0', grad_fn=<SliceBackward0>)

Layer: linear_relu_stack.0.bias|Size: torch.Size([512])|Values: tensor([-0.0341,  0.0170], device='cuda:0', grad_fn=<SliceBackward0>)

Layer: linear_relu_stack.2.weight|Size: torch.Size([512, 512])|Values: tensor([[ 0.0274,  0.0349,  0.0196,  ..., -0.0113, -0.0057, -0.0176],
        [ 0.0361, -0.0430,  0.0207,  ..., -0.0360, -0.0158,  0.0425]],
       device='cuda:0', grad_fn=<SliceBackward0>)

L